<a href="https://colab.research.google.com/github/OscarPellicer/extra-attributions/blob/main/notebooks/03_llm_interpretability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 · Hands-on: looking inside an LLM

We look inside **GPT-2 small** (12 layers, residual stream of dimension 768) with [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens) and [SAELens](https://github.com/jbloomAus/SAELens):

1. **Logit lens**: read the intermediate layers as if they were the output.
2. **Linear probes**: is sentiment linearly decodable at each layer? (the idea behind TCAV)
3. **Sparse autoencoder (SAE) features**, inspected on [Neuronpedia](https://www.neuronpedia.org).
4. **Steering**: add a feature direction and watch the generations change.

Parameters marked `# change me` list values worth trying. The first run downloads GPT-2 (~500 MB) and the SAE (~300 MB).

*A longer version with exercises and answers is in `extended/03_llm_interpretability.ipynb`.*

In [ ]:
import sys

if 'google.colab' in sys.modules:
    # Colab: install the two interpretability libraries (torch, sklearn, matplotlib... are preinstalled)
    !pip install -q sae-lens "transformer-lens<4"
else:
    print("Running locally: make sure you installed the requirements, e.g.\n"
          "    pip install -r requirements.txt")

In [ ]:
import os, warnings
os.environ["TRANSFORMERLENS_ALLOW_MPS"] = "1"  # we check MPS correctness ourselves below
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import random
from functools import partial

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML, IFrame

from transformer_lens import HookedTransformer
from sae_lens import SAE

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(0)
torch.set_grad_enabled(False)  # we never need gradients in this notebook

# Device: cuda > mps > cpu
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using {device=}")

In [ ]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()

# Sanity check: some PyTorch versions give silently wrong results on Apple MPS -> compare against CPU, fall back if needed
if device == "mps":
    test = "The quick brown fox jumps over the lazy"
    try:
        logits_mps = model(test).float().cpu()
        model.to("cpu"); logits_cpu = model(test).float()
        ok = torch.allclose(logits_mps, logits_cpu, atol=1e-2)
    except Exception as e:
        print("MPS error:", e); ok = False
    if ok:
        model.to("mps")
    else:
        device = "cpu"
        print("MPS results differ from CPU (or MPS failed): falling back to CPU")
print(f"Model on {device}: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}, vocab={model.cfg.d_vocab}")

In [ ]:
PROMPT = "The Colosseum is located in the city of"  # change me
TOP_K = 10                                          # change me

def show_top_predictions(prompt, k=TOP_K):
    logits = model(prompt)                       # [batch=1, n_tokens, vocab]
    probs = logits[0, -1].softmax(-1)            # distribution over the NEXT token
    p, idx = probs.topk(k)
    print(f"Tokens: {model.to_str_tokens(prompt)}")
    for rank, (pi, ti) in enumerate(zip(p, idx)):
        print(f"  {rank:2d}. {model.to_string(ti)!r:15} {100 * pi.item():5.1f}%")

show_top_predictions(PROMPT)

## 1. Logit lens

Apply the final LayerNorm and the unembedding to the residual stream after each layer: *when* does the prediction become readable? (Try the Eiffel Tower / `' Paris'`, or `"The Statue of Liberty is in New"` / `' York'`.)

In [ ]:
PROMPT = "The Colosseum is located in the city of"  # change me
TARGET = " Rome"                                    # change me (note the leading space!)

target_id = model.to_single_token(TARGET)
_, cache = model.run_with_cache(PROMPT)

# Residual stream at the last position before each layer + final output: [n_layers+1, d_model]
resid, labels = cache.accumulated_resid(apply_ln=True, pos_slice=-1, return_labels=True)
layer_logits = model.unembed(resid)[:, 0]          # [n_layers+1, vocab]
layer_probs = layer_logits.softmax(-1)

ranks = (layer_logits > layer_logits[:, [target_id]]).sum(-1).cpu().numpy()
probs = layer_probs[:, target_id].cpu().numpy()
top1 = [model.to_string(i) for i in layer_logits.argmax(-1)]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(ranks + 1, "o-"); ax[0].set_yscale("log"); ax[0].invert_yaxis()
ax[0].set_title(f"Rank of {TARGET!r} (1 = top prediction)"); ax[0].set_xlabel("layer")
ax[1].plot(probs, "o-", color="C1"); ax[1].set_title(f"Probability of {TARGET!r}"); ax[1].set_xlabel("layer")
for a in ax:
    a.set_xticks(range(len(labels))); a.set_xticklabels(range(len(labels))); a.grid(alpha=.3)
plt.suptitle(f"Logit lens: {PROMPT!r}"); plt.tight_layout(); plt.show()

print("Top-1 token decoded at each layer (layer 12 = model output):")
print(pd.DataFrame({"top-1 token": top1, f"rank of {TARGET!r}": ranks + 1, "p": probs.round(3)}).T.to_string())

## 2. Linear probes

We collect the residual stream of short sentences (*"The movie was great."*) at every layer and train a logistic regression to predict their sentiment, testing on **adjectives never seen in training**. The **control** task (a random label per word) shows how much a probe can "learn by itself".

In [ ]:
POSITIVE = ["great", "wonderful", "excellent", "amazing", "fantastic", "lovely", "brilliant", "delightful",
            "superb", "pleasant", "enjoyable", "beautiful", "perfect", "charming", "fun", "awesome"]
NEGATIVE = ["terrible", "awful", "horrible", "dreadful", "boring", "disappointing", "bad", "poor",
            "miserable", "unpleasant", "annoying", "ugly", "disgusting", "mediocre", "painful", "useless"]
TEMPLATES = ["The movie was {}.", "I thought the food was {}.", "Honestly, the service was {}.",
             "My day was {}.", "The concert last night was {}.", "Overall, the book is {}."]
INCLUDE_NEGATION = False   # change me: True also adds "... was not {}." sentences with the label flipped

sentences, labels, words = [], [], []
for t in TEMPLATES:
    variants = [(t, False)] + ([(t.replace(" {}", " not {}"), True)] if INCLUDE_NEGATION else [])
    for tt, negated in variants:
        for w in POSITIVE: sentences.append(tt.format(w)); labels.append(int(not negated)); words.append(w)
        for w in NEGATIVE: sentences.append(tt.format(w)); labels.append(int(negated)); words.append(w)
labels, words = np.array(labels), np.array(words)
print(len(sentences), "sentences, e.g.:", sentences[0], sentences[16], sentences[-1])

In [ ]:
PROBE_POSITION = -1   # change me: -1 = final '.', -2 = last token of the adjective

layers = list(range(model.cfg.n_layers + 1))                # 0 = embeddings, l = after block l-1
hooks = [f"blocks.{l}.hook_resid_pre" for l in range(model.cfg.n_layers)] + \
        [f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"]
acts = np.zeros((len(layers), len(sentences), model.cfg.d_model), dtype=np.float32)
for i, s in enumerate(sentences):
    _, c = model.run_with_cache(s, names_filter=lambda n: n in hooks)
    for l, h in enumerate(hooks):
        acts[l, i] = c[h][0, PROBE_POSITION].float().cpu().numpy()
print("Activations:", acts.shape, "(layers, sentences, d_model)")

In [ ]:
SPLIT_BY_WORD = True   # change me: True = test on held-out adjectives, False = random split of sentences
C = 0.1                # change me: inverse regularization strength of the logistic regression

rng = np.random.default_rng(0)
if SPLIT_BY_WORD:
    test_words = set(rng.choice(POSITIVE, 5, replace=False)) | set(rng.choice(NEGATIVE, 5, replace=False))
    test = np.isin(words, list(test_words))
else:
    test = rng.random(len(sentences)) < 0.3
train = ~test
# Control task (Hewitt & Liang, 2019): each adjective gets a RANDOM label, unrelated to its sentiment
all_words = POSITIVE + NEGATIVE
random_word_label = dict(zip(all_words, rng.permutation([0, 1] * (len(all_words) // 2))))
control_labels = np.array([random_word_label[w] for w in words])

def probe_acc(X, y):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    clf.fit(X[train], y[train])
    return clf.score(X[test], y[test])

acc_real = [probe_acc(acts[l], labels) for l in layers]
acc_ctrl = [probe_acc(acts[l], control_labels) for l in layers]

plt.figure(figsize=(7, 3.5))
plt.plot(layers, acc_real, "o-", label="sentiment labels")
plt.plot(layers, acc_ctrl, "s--", label="control: random label per word")
plt.axhline(0.5, color="gray", lw=1, ls=":", label="chance")
plt.xlabel("layer (0 = embeddings)"); plt.ylabel("test accuracy"); plt.ylim(0, 1.05); plt.xticks(layers)
plt.title(f"Linear probe, position {PROBE_POSITION}, split_by_word={SPLIT_BY_WORD}")
plt.legend(); plt.grid(alpha=.3); plt.show()
print("real:", np.round(acc_real, 2)); print("ctrl:", np.round(acc_ctrl, 2))

## 3. Sparse autoencoder features

An SAE re-expresses each 768-d activation as a sparse combination of **24,576 features**, many of them interpretable. We use a pretrained SAE at the input of layer 8.

In [ ]:
SAE_RELEASE, SAE_LAYER = "gpt2-small-res-jb", 8
HOOK = f"blocks.{SAE_LAYER}.hook_resid_pre"

sae = SAE.from_pretrained(SAE_RELEASE, HOOK, device=device)
if isinstance(sae, tuple):  # sae-lens < 6 returned (sae, cfg_dict, sparsity)
    sae = sae[0]
sae.eval()
print(f"SAE: d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}  ->  {sae.cfg.d_sae / sae.cfg.d_in:.0f}x overcomplete")

In [ ]:
TEXT = ("After the ceremony in the old chapel, the bride and groom walked down the aisle while the guests "
        "threw rice. The reception was held in a garden and everyone danced until midnight.")  # change me

tokens = model.to_tokens(TEXT)                      # prepends <|endoftext|> (BOS), as the SAE was trained with it
str_tokens = model.to_str_tokens(TEXT)
_, cache = model.run_with_cache(tokens, names_filter=HOOK)
x = cache[HOOK][0]                                  # [n_tokens, 768]
feats = sae.encode(x)                               # [n_tokens, 24576]
x_hat = sae.decode(feats)

l0 = (feats[1:] > 0).sum(-1).float()                # skip BOS (huge norm, atypical)
fvu = ((x[1:] - x_hat[1:]) ** 2).sum() / ((x[1:] - x[1:].mean(0)) ** 2).sum()
print(f"L0 (active features per token): mean {l0.mean():.1f} of {sae.cfg.d_sae}  ({100 * l0.mean() / sae.cfg.d_sae:.2f}%)")
print(f"Variance explained by the reconstruction: {100 * (1 - fvu):.1f}%")

# Does the model still work if we replace the residual stream by the SAE reconstruction? (CE loss)
def splice_sae(act, hook):    return sae.decode(sae.encode(act)).to(act.dtype)
def zero_ablate(act, hook):   return torch.zeros_like(act)
ce_clean = model(tokens, return_type="loss").item()
ce_sae   = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(HOOK, splice_sae)]).item()
ce_zero  = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(HOOK, zero_ablate)]).item()
print(f"Next-token CE loss: clean {ce_clean:.2f} | SAE spliced in {ce_sae:.2f} | zero-ablated {ce_zero:.2f}")
print(f"-> CE loss recovered: {100 * (ce_zero - ce_sae) / (ce_zero - ce_clean):.1f}%")

In [ ]:
N_TOP_FEATURES = 3   # change me

vals, idx = feats.topk(N_TOP_FEATURES, dim=-1)
rows = []
for i, tok in enumerate(str_tokens):
    rows.append({"token": repr(tok), **{f"feat #{k+1}": f"{idx[i, k].item()} ({vals[i, k].item():.1f})"
                                       for k in range(N_TOP_FEATURES)}})
display(pd.DataFrame(rows).iloc[1:])   # skip BOS

Feature **5897** is the top feature on `bride` and `groom`. Where does it fire on other texts? Does its Neuronpedia description match?

In [ ]:
FEATURE = 5897   # change me (try other indices from the table above)

TEXTS = [
    "They got married last June; the wedding ceremony was small and the bride wore her mother's gown.",
    "We sent the invitations for our anniversary party and booked a band for the reception.",
    "The groom's best man gave a funny speech and then they cut the cake.",
    "The stock market fell sharply on Monday after the central bank raised interest rates.",
    "To make the sauce, fry the garlic in olive oil and add the chopped tomatoes.",
    "The astronauts docked with the space station after a six-hour flight.",
    "The striker scored twice in the second half and the team won the championship.",
    "Python lists are mutable, while tuples cannot be changed after creation.",
    "Santa brought presents and we sang carols around the Christmas tree.",
    "The funeral procession moved slowly through the village towards the church.",
]

def feature_acts(text, feature):
    toks = model.to_tokens(text)
    _, c = model.run_with_cache(toks, names_filter=HOOK)
    return model.to_str_tokens(text)[1:], sae.encode(c[HOOK][0, 1:])[:, feature].cpu().numpy()

def html_highlight(str_toks, acts, vmax):
    spans = "".join(f'<span style="background: rgba(255,120,0,{min(a / vmax, 1):.2f}); color: black" '
                    f'title="{a:.2f}">{t.replace("<", "&lt;")}</span>' for t, a in zip(str_toks, acts))
    return f'<div style="font-family: monospace; line-height: 1.8; background: white; padding: 2px">{spans}</div>'

results = [feature_acts(t, FEATURE) for t in TEXTS]
vmax = max(max(a.max() for _, a in results), 1e-6)
display(HTML(f"<b>Feature {FEATURE}</b> (orange = active, hover for value)" +
             "".join(html_highlight(s, a, vmax) for s, a in results)))

# Top-activating tokens across these texts
all_acts = [(a, s[j], i) for i, (s, acts) in enumerate(results) for j, a in enumerate(acts)]
print("Top activating tokens:")
for a, tok, i in sorted(all_acts, reverse=True)[:10]:
    print(f"  {a:6.2f}  {tok!r:15} (text {i})")

url = f"https://www.neuronpedia.org/gpt2-small/{SAE_LAYER}-res-jb/{FEATURE}"
print(f"\nInspect this feature on Neuronpedia: {url}")

## 4. Feature steering

Adding $\alpha \cdot d_f$ (the feature's decoder direction) to the residual stream at every position should make the model "think" about the concept (cf. *Golden Gate Claude*). Unlike a probe, this is a **causal** test.

In [ ]:
STEER_FEATURE = 5897                 # change me (5897 weddings; try 14149 recipes, 3313 space missions, 8526 Christmas)
STEER_COEF = 60                      # change me (try 0, 20, 60, 150, 400, -60)
STEER_PROMPT = "I think that"        # change me
N_SAMPLES, MAX_NEW_TOKENS, SEED = 3, 30, 0

steer_dir = sae.W_dec[STEER_FEATURE].detach().clone()

def steering_hook(act, hook, coef):
    return act + coef * steer_dir.to(act.dtype)

def generate(prompt, coef, seed=SEED):
    set_seed(seed)
    with model.hooks(fwd_hooks=[(HOOK, partial(steering_hook, coef=coef))]):
        return model.generate(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=0.7, top_p=0.9,
                              freq_penalty=1.0, verbose=False)

print(f"Feature {STEER_FEATURE}: https://www.neuronpedia.org/gpt2-small/{SAE_LAYER}-res-jb/{STEER_FEATURE}\n")
for s in range(N_SAMPLES):
    print(f"--- sample {s} ---")
    print("  normal :", repr(generate(STEER_PROMPT, 0, seed=SEED + s)))
    print("  steered:", repr(generate(STEER_PROMPT, STEER_COEF, seed=SEED + s)))

## Takeaways

- The **logit lens** shows *when* a prediction forms (early layers are not yet in "output space").
- **Probes** show what is *decodable*, need controls and careful splits, and are not evidence that the model *uses* it.
- **SAEs** give a sparse, often interpretable basis, with imperfect reconstruction.
- **Steering** is causal: if adding a direction changes the behaviour as expected, the model uses it.

To go further: attribution graphs on [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/graph) and [`circuit-tracer`](https://github.com/safety-research/circuit-tracer).